##### Copyright 2020 The TensorFlow Authors.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# TensorFlow Recommenders: Quickstart

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/recommenders/quickstart"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/recommenders/blob/main/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/recommenders/blob/main/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/recommenders/docs/examples/quickstart.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

In this tutorial, we build a simple matrix factorization model using the [MovieLens 100K dataset](https://grouplens.org/datasets/movielens/100k/) with TFRS. We can use this model to recommend movies for a given user.

### Import TFRS

First, install and import TFRS:

In [2]:
!pip install -q tensorflow-recommenders
!pip install -q --upgrade tensorflow-datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 929.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 30.9 MB/s eta 0:00:00


In [3]:
from typing import Dict, Text

import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds
import tensorflow_recommenders as tfrs

### Read the data

In [4]:
# Ratings data.
ratings = tfds.load('movielens/100k-ratings', split="train")
# Features of all the available movies.
movies = tfds.load('movielens/100k-movies', split="train")

# Select the basic features.
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"]
})
movies = movies.map(lambda x: x["movie_title"])

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/movielens/100k-movies/incomplete.H3GGQO_0.1.1/movielens-train.tfrecord*...…

Dataset movielens downloaded and prepared to /root/tensorflow_datasets/movielens/100k-movies/0.1.1. Subsequent calls will reuse this data.


In [5]:
len(ratings)
len(movies)

1682

In [6]:
for r in ratings.take(10):  # Change 5 to however many you want
    print(r)

{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b"One Flew Over the Cuckoo's Nest (1975)">, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'138'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Strictly Ballroom (1992)'>, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'92'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Very Brady Sequel, A (1996)'>, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'301'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Pulp Fiction (1994)'>, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'60'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Scream 2 (1997)'>, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'197'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Crash (1996)'>, 'user_id': <tf.Tensor: shape=(), dtype=string, numpy=b'601'>}
{'movie_title': <tf.Tensor: shape=(), dtype=string, numpy=b'Aladdin (1992)'>, 'user_id': <tf.Tensor: shape=(), 

In [7]:
for r in movies.take(5):  # Change 5 to however many you want tp print
    print(r)
    print(type(r))

tf.Tensor(b'You So Crazy (1994)', shape=(), dtype=string)
<class 'tensorflow.python.framework.ops.EagerTensor'>
tf.Tensor(b'Love Is All There Is (1996)', shape=(), dtype=string)
<class 'tensorflow.python.framework.ops.EagerTensor'>
tf.Tensor(b'Fly Away Home (1996)', shape=(), dtype=string)
<class 'tensorflow.python.framework.ops.EagerTensor'>
tf.Tensor(b'In the Line of Duty 2 (1987)', shape=(), dtype=string)
<class 'tensorflow.python.framework.ops.EagerTensor'>
tf.Tensor(b'Niagara, Niagara (1997)', shape=(), dtype=string)
<class 'tensorflow.python.framework.ops.EagerTensor'>


Build vocabularies to convert user ids and movie titles into integer indices for embedding layers:

In [ ]:
# creating vocab for user and movie embedding matrix separately. string lookup converts the strings into the integer indices or numpy indices. It only adds the index not necessarily convert to integer

user_ids_vocabulary = tf.keras.layers.StringLookup(mask_token=None)
user_ids_vocabulary.adapt(ratings.map(lambda x: x["user_id"]))

movie_titles_vocabulary = tf.keras.layers.StringLookup(mask_token=None)
movie_titles_vocabulary.adapt(movies)

In [9]:
# why there are only 6 user id's

len(user_ids_vocabulary.get_vocabulary())

944

In [10]:
print(user_ids_vocabulary.get_vocabulary()[:5])

['[UNK]', np.str_('405'), np.str_('655'), np.str_('13'), np.str_('450')]


In [11]:
len(movie_titles_vocabulary.get_vocabulary())

1665

In [12]:
print(movie_titles_vocabulary.get_vocabulary()[:5])

['[UNK]', np.str_("Ulee's Gold (1997)"), np.str_('That Darn Cat! (1997)'), np.str_('Substance of Fire, The (1996)'), np.str_('Sliding Doors (1998)')]


### Define a model

We can define a TFRS model by inheriting from `tfrs.Model` and implementing the `compute_loss` method:

In [13]:
class MovieLensModel(tfrs.Model):
  # We derive from a custom base class to help reduce boilerplate. Under the hood,
  # these are still plain Keras Models.

  def __init__(
      self,
      user_model: tf.keras.Model,
      movie_model: tf.keras.Model,
      task: tfrs.tasks.Retrieval):
    super().__init__()

    # Set up user and movie representations.
    self.user_model = user_model
    self.movie_model = movie_model

    # Set up a retrieval task.
    self.task = task

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    # Define how the loss is computed.

    user_embeddings = self.user_model(features["user_id"])
    movie_embeddings = self.movie_model(features["movie_title"])

    return self.task(user_embeddings, movie_embeddings)

Define the two models and the retrieval task.

In [14]:
# Define user embedding model, by passing unique user id vocab and embedding size of of 64, so embedding size is 944*64

user_model = tf.keras.Sequential([
    user_ids_vocabulary,
    tf.keras.layers.Embedding(len(user_ids_vocabulary.get_vocabulary()), 64)
])

In [15]:
print(user_model.layers)

[<StringLookup name=string_lookup, built=False>, <Embedding name=embedding, built=False>]


In [16]:
# define movies embedding model, by passing movie vocab and 64 as embedding size so embedding size is 1665 * 64

movie_model = tf.keras.Sequential([
    movie_titles_vocabulary,
    tf.keras.layers.Embedding(len(movie_titles_vocabulary.get_vocabulary()), 64)
])

In [26]:
print(movie_model.layers)

[<StringLookup name=string_lookup_1, built=True>, <Embedding name=embedding_1, built=True>]


In [24]:
# Access the Embedding layer inside the Sequential model
embedding_layer = movie_model.layers[1]

# Explicitly build the movie_model or pass some data through it to initialize weights
# We can pass a movie title to build the model
movie_model(tf.constant("You So Crazy (1994)"))

# Get the weights (embeddings) from the Embedding layer
embedding_weights = embedding_layer.get_weights()[0]

# Print the full embedding matrix (can be large!)
print(embedding_weights)

# Or, print the shape of the matrix
print("Embedding shape:", embedding_weights.shape)


[[ 0.03060546  0.04367626  0.04413059 ...  0.04910752  0.01243043
  -0.00590743]
 [-0.02169858 -0.01731242  0.0011657  ...  0.0415145   0.00771431
  -0.0479016 ]
 [-0.02456473  0.00071608  0.01784955 ... -0.04809372 -0.00821199
   0.03011471]
 ...
 [-0.03149203 -0.0372551   0.04588895 ...  0.01571557  0.01338587
   0.03371116]
 [ 0.0074093   0.02780632 -0.00950625 ... -0.00770348  0.02496092
  -0.00582694]
 [-0.0016802  -0.01737111  0.00116298 ... -0.01329412  0.0115528
   0.01393947]]
Embedding shape: (1665, 64)


In [23]:
movies

<_MapDataset element_spec=TensorSpec(shape=(), dtype=tf.string, name=None)>

In [25]:
# Define your objectives - retrieval task,, what i think is happening here is, the embedding matrix is1665*64, i think embedding size is 64 and vocab is 1665. But the embedding model is empty and movies i.e titles is not numeric/numpy/tensor ?
# But if i print embedding model for movie, it is printing it, it is tensor

task = tfrs.tasks.Retrieval(metrics=tfrs.metrics.FactorizedTopK(
    movies.batch(128).map(movie_model)
  )
)

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("args_0:0", shape=(None,), dtype=string). Expected shape (), but input has incompatible shape (None,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None,), dtype=string)
  • training=None
  • mask=None


### Fit and evaluate it.

Create the model, train it, and generate predictions:



In [ ]:
# # Create a retrieval model.
# model = MovieLensModel(user_model, movie_model, task)
# model.compile(optimizer=tf.keras.optimizers.Adagrad(0.5))

# # Train for 3 epochs.
# model.fit(ratings.batch(4096), epochs=3)

# # Use brute-force search to set up retrieval using the trained representations.
# index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
# index.index_from_dataset(
#     movies.batch(100).map(lambda title: (title, model.movie_model(title))))

# # Get some recommendations.
# _, titles = index(np.array(["42"]))
# print(f"Top 3 recommendations for user 42: {titles[0, :3]}")